In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import numpy as np
import pandas as pd
import pickle

df_cleaned_resume = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_resume.pkl')
df_cleaned_jd = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_jd.pkl')

similarity_matrix = np.load('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/similarity_matrix.npy')

with open('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/tfidf_vectorizer.pkl', 'rb') as f:
  vectorizer = pickle.load(f)

In [23]:
print(df_cleaned_jd.columns.tolist())
print(df_cleaned_resume.columns.tolist())
print(df_cleaned_jd.shape)
print(df_cleaned_resume.shape)
print(similarity_matrix.shape)

['category', 'job_title', 'job_description', 'cleaned_description', 'extracted_skills']
['ID', 'Resume_str', 'Resume_html', 'Category', 'cleaned_resume', 'extracted_skills', 'best_match_category', 'best_match_score', 'best_match_category_combined', 'best_match_combined']
(6, 5)
(697, 10)
(697, 6)


In [24]:
category_order = list(df_cleaned_jd['category'])
for cat in category_order:
    skills = df_cleaned_jd[df_cleaned_jd['category']==cat]['extracted_skills'].iloc[0]
    print(cat, ':', skills, '| count:', len(skills))

INFORMATION-TECHNOLOGY : ['microsoft', 'technical', 'desktop', 'application'] | count: 4
FINANCE : ['finance', 'accountant', 'balance', 'cash', 'reconciliation', 'variance', 'forecast', 'revenue', 'audit', 'accounting', 'financial statement'] | count: 11
ENGINEERING : ['material', 'inspection', 'production', 'drawing', 'mechanical', 'manufacturing'] | count: 6
SALES : ['customer service', 'stock', 'sell'] | count: 3
HR : ['generalist', 'employee relation', 'human resource', 'compensation', 'interview', 'benefit', 'recruit'] | count: 7
HEALTHCARE : ['insurance', 'treatment', 'patient', 'provider', 'hospital', 'nursing', 'nurse'] | count: 7


In [25]:
def identify_missing_skills(resume_skills, jd_skills):
  missing = set(jd_skills) - set(resume_skills)
  return list(missing)

In [26]:
sample_idx = df_cleaned_resume[df_cleaned_resume['Category'] == 'FINANCE'].index[0]
resume_skills = df_cleaned_resume.loc[sample_idx, 'extracted_skills']
jd_skills = df_cleaned_jd[df_cleaned_jd['category'] == 'FINANCE']['extracted_skills'].iloc[0]

print("resume skills :", resume_skills)
print("jd_requires :", jd_skills)
print("missing skills :", identify_missing_skills(resume_skills, jd_skills))

resume skills : ['finance', 'transaction', 'cash', 'variance', 'forecast', 'audit', 'operating', 'interview', 'budgeting', 'accounting', 'background', 'bank', 'store']
jd_requires : ['finance', 'accountant', 'balance', 'cash', 'reconciliation', 'variance', 'forecast', 'revenue', 'audit', 'accounting', 'financial statement']
missing skills : ['financial statement', 'accountant', 'revenue', 'reconciliation', 'balance']


In [27]:
def skill_overlap_score(resume_skills, jd_skills):
  if (len(jd_skills) == 0):
    return 0
  overlap = set(resume_skills) & set(jd_skills)
  overlap_score = len(overlap)/len(jd_skills)
  return overlap_score

In [28]:
def full_candidate_report(jd_category, top_n):
  jd_idx = list(df_cleaned_jd['category']).index(jd_category)
  jd_skills = df_cleaned_jd.iloc[jd_idx]['extracted_skills']

  results = df_cleaned_resume.copy()
  results['similarity_score'] = similarity_matrix[:,jd_idx]
  results['skill_overlap'] = results['extracted_skills'].apply(lambda skills: skill_overlap_score(skills, jd_skills))
  results['combined_score'] = (0.7 * results['similarity_score']) + (0.3 * results['skill_overlap'])
  results['missing_skills'] = results['extracted_skills'].apply(lambda skills: identify_missing_skills(skills, jd_skills))

  min_score = results['combined_score'].min()
  max_score = results['combined_score'].max()

  results['match_percentage'] = ((results['combined_score'] - min_score) / (max_score - min_score) * 100).round(1)

  ranked = results.sort_values('combined_score', ascending=False)
  return ranked[['Category', 'match_percentage', 'missing_skills']].head(top_n)

In [29]:
full_candidate_report('FINANCE', 7)

,Category,match_percentage,missing_skills
1493,FINANCE,100.0,[variance]
1524,FINANCE,95.1,[forecast]
1557,FINANCE,94.9,[accountant]
1505,FINANCE,94.4,[]
1483,FINANCE,94.2,[accountant]
1585,FINANCE,91.7,[balance]
1491,FINANCE,90.9,"[audit, revenue]"


In [30]:
full_candidate_report('HEALTHCARE', 7)

,Category,match_percentage,missing_skills
761,HEALTHCARE,100.0,[provider]
791,HEALTHCARE,97.3,"[provider, insurance]"
745,HEALTHCARE,95.7,"[provider, insurance]"
783,HEALTHCARE,92.3,"[provider, insurance]"
693,HEALTHCARE,86.2,[provider]
786,HEALTHCARE,82.7,[provider]
731,HEALTHCARE,82.5,[nurse]


In [31]:
full_candidate_report('INFORMATION-TECHNOLOGY', 7)

,Category,match_percentage,missing_skills
281,INFORMATION-TECHNOLOGY,100.0,[]
234,INFORMATION-TECHNOLOGY,92.3,[]
1091,SALES,92.3,[]
225,INFORMATION-TECHNOLOGY,91.5,[]
283,INFORMATION-TECHNOLOGY,91.5,[]
223,INFORMATION-TECHNOLOGY,91.2,[]
258,INFORMATION-TECHNOLOGY,91.0,[]


In [32]:
full_candidate_report('SALES', 7)

,Category,match_percentage,missing_skills
1085,SALES,100.0,[]
1014,SALES,99.3,[]
1088,SALES,98.5,[]
1081,SALES,97.8,[]
1078,SALES,96.7,[]
1080,SALES,96.2,[]
1016,SALES,95.8,[]


In [33]:
full_candidate_report('HR', 7)

,Category,match_percentage,missing_skills
11,HR,100.0,[]
23,HR,99.6,[]
81,HR,98.3,[compensation]
67,HR,97.3,[]
85,HR,97.2,[]
19,HR,96.9,[]
4,HR,96.7,[]


In [34]:
full_candidate_report('ENGINEERING', 7)

,Category,match_percentage,missing_skills
1776,ENGINEERING,100.0,[]
1787,ENGINEERING,97.6,[]
1084,SALES,94.8,[]
1803,ENGINEERING,91.9,[mechanical]
1737,ENGINEERING,91.1,[]
1720,ENGINEERING,89.8,[production]
1709,ENGINEERING,87.8,[material]


In [35]:
def full_candidate_report_tail(jd_category, tail_n):
  jd_idx = list(df_cleaned_jd['category']).index(jd_category)
  jd_skills = df_cleaned_jd.iloc[jd_idx]['extracted_skills']

  results = df_cleaned_resume.copy()
  results['similarity_score'] = similarity_matrix[:,jd_idx]
  results['skill_overlap'] = results['extracted_skills'].apply(lambda skills: skill_overlap_score(skills, jd_skills))
  results['combined_score'] = (0.7 * results['similarity_score']) + (0.3 * results['skill_overlap'])
  results['missing_skills'] = results['extracted_skills'].apply(lambda skills: identify_missing_skills(skills, jd_skills))

  min_score = results['combined_score'].min()
  max_score = results['combined_score'].max()

  results['match_percentage'] = ((results['combined_score'] - min_score) / (max_score - min_score) * 100).round(1)

  ranked = results.sort_values('combined_score', ascending=False)
  return ranked[['Category', 'match_percentage', 'missing_skills']].tail(tail_n)

In [36]:
full_candidate_report_tail('INFORMATION-TECHNOLOGY', 10)

,Category,match_percentage,missing_skills
1026,SALES,2.4,"[microsoft, technical, application, desktop]"
1033,SALES,2.3,"[microsoft, technical, application, desktop]"
1049,SALES,2.2,"[microsoft, technical, application, desktop]"
736,HEALTHCARE,1.6,"[microsoft, technical, application, desktop]"
1051,SALES,1.5,"[microsoft, technical, application, desktop]"
1806,ENGINEERING,1.2,"[microsoft, technical, application, desktop]"
1044,SALES,1.1,"[microsoft, technical, application, desktop]"
705,HEALTHCARE,1.1,"[microsoft, technical, application, desktop]"
1770,ENGINEERING,0.8,"[microsoft, technical, application, desktop]"
1102,SALES,0.0,"[microsoft, technical, application, desktop]"


In [37]:
full_candidate_report_tail('SALES', 10)

,Category,match_percentage,missing_skills
298,INFORMATION-TECHNOLOGY,1.5,"[customer service, stock, sell]"
255,INFORMATION-TECHNOLOGY,1.3,"[customer service, stock, sell]"
263,INFORMATION-TECHNOLOGY,1.2,"[customer service, stock, sell]"
1547,FINANCE,1.2,"[customer service, stock, sell]"
1773,ENGINEERING,1.2,"[customer service, stock, sell]"
227,INFORMATION-TECHNOLOGY,1.0,"[customer service, stock, sell]"
1717,ENGINEERING,1.0,"[customer service, stock, sell]"
1715,ENGINEERING,0.9,"[customer service, stock, sell]"
271,INFORMATION-TECHNOLOGY,0.6,"[customer service, stock, sell]"
1786,ENGINEERING,0.0,"[customer service, stock, sell]"


**manual validation**

In [38]:
healthcare_top5 = full_candidate_report('HEALTHCARE', 5)

In [39]:
for idx in healthcare_top5.index[:3]:
    print(f"--- Resume {idx} (match: {healthcare_top5.loc[idx, 'match_percentage']}%) ---")
    print(df_cleaned_resume.loc[idx, 'Resume_str'][:600])
    print()

--- Resume 761 (match: 100.0%) ---
         OCCUPATIONAL HEALTH NURSE COORDINATOR           Professional Summary    Dedicated RN with over 20 years experience in nursing seeking career transition into a new clinical setting. Able to offer a solid foundation in occupational health, office management, triage, endoscopy, homecare, primary patient care and current healthcare advancements. Highly motivated, dedicated, flexible and compassionate with proven expertise in communication, organization and documentation skills. Valuable interpersonal skills, forging relationships and collaborating with interdisciplinary teams to develop o

--- Resume 791 (match: 97.3%) ---
         LICENSED PRACTICAL NURSE- STEP-DOWN UNIT       Summary       Licensed Practical Nurse with 15 years in providing direct care under RN and MD supervision in diagnosis, treatment prescription and follow-up with patients from pediatrics to geriatrics. Additional expertise includes management and staff supervision. Strong 

In [40]:
print(df_cleaned_resume['Category'].value_counts())

Category
INFORMATION-TECHNOLOGY    120
ENGINEERING               118
FINANCE                   118
SALES                     116
HEALTHCARE                115
HR                        110
Name: count, dtype: int64


In [41]:
for cat in ['SALES', 'INFORMATION-TECHNOLOGY']:
    report = full_candidate_report(cat, 3)
    for idx in report.index:
        print(f"--- {cat} - Resume {idx} (match: {report.loc[idx, 'match_percentage']}%) ---")
        print(df_cleaned_resume.loc[idx, 'Resume_str'][:500])
        print()

--- SALES - Resume 1085 (match: 100.0%) ---
         SALES ASSOCIATE/MERCHANDISER       Experience      Sales Associate/Merchandiser     Jul 2014   to   Current      Company Name         Investigate and resolved customer inquiries and complaints in a timely and empathetic
          manner.  Run markdown reports, manage store replenishment and analyze buying reports.  Contact customers to follow up on purchases, suggest new merchandise and inform them about
          promotions and upcoming events.  Operate POS system to itemize, open cred

--- SALES - Resume 1014 (match: 99.3%) ---
         SALES       Summary    To obtain a position where I can utilize my skills and work in an environment that will enhance my knowledge and career. Great organization and communication skills that will aid in excellent customer service and satisfaction.      Highlights          Bi-lingual   Multi-line system expert  Superior communication skills  Data entry  Claims expert  Install coordinator  Proficien